# IBI36 filtré et interpolé : comparaison avec GLO12

Température proche de la surface, le **3 janvier 2024**, dans le golfe de Gascogne. **IBI36 au 1/36° → filtre de Shapiro → grille GLO12 au 1/12°.**

In [ ]:
import numpy as np
import xarray as xr
from oceanbench.datasets.reference import ibi36_reference, glo12_analysis
from assets.shapiro import shapiro_1d

day = "2024-01-03"

## Lire IBI36 et appliquer le filtre

On appelle la fonction d'origine `shapiro_1d` sur une seule carte. Le nombre de passages se règle avec `id_np`.

In [ ]:
temperature = ibi36_reference(day, day).thetao.sel(
    latitude=slice(44, 48), longitude=slice(-8, -2)
).isel(time=0, depth=0, drop=True)

filtered = shapiro_1d(temperature.load().values[None, :, :], id_np=15, missval=np.nan, ln_depth=False)[0]
temperature_filtered = temperature.copy(data=filtered)

## Interpoler vers GLO12

On lit GLO12 le même jour et sur la même zone, près de la surface. `interp_like` interpole directement la carte filtrée sur la grille GLO12, avec une interpolation linéaire. Les points hors couverture restent masqués.

In [ ]:
glo12_temperature = glo12_analysis().thetao.sel(
    time=slice(day, day), latitude=slice(44, 48), longitude=slice(-8, -2)
).isel(time=0, depth=0, drop=True).load()

temperature_1_12 = temperature_filtered.interp_like(glo12_temperature)

## Comparaison visuelle

IBI36 filtré et GLO12 sont affichés côte à côte, sur la même grille, avec la même échelle de couleurs.

In [ ]:
comparison = xr.concat([temperature_1_12, glo12_temperature], dim="champ")
comparison["champ"] = ["IBI36 filtre", "GLO12"]
comparison.plot(col="champ", cmap="turbo");